SIGNAL TEST

The goal of this section is to add a signal component to the model and get an estimate for the posterior distribution p(mu_s|data). After that we shall proceed with model comparison. 

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.integrate import trapezoid

THE CODE BELOW IS A RELOAD OF ALL THE IMPORTANT VARIABLES

In [3]:
bins = pd.read_csv('SourceData/s2_binning_info.csv')
resp_nr = pd.read_csv('SourceData/s2_response_nr.csv')
resp_er = pd.read_csv('SourceData/s2_response_er.csv')
bg = pd.read_csv('SourceData/er_and_cevns_background.csv')
events = pd.read_csv('SourceData/events_after_cuts.csv')

In [4]:
s2_bin_centers_log = bins['log_center_pe'].values
s2_bin_centers_lin = bins['linear_center_pe'].values
s2_bin_widths = (bins['end_pe'] - bins['start_pe']).values
s2_bin_edges = np.concatenate([bins['start_pe'].values, [bins['end_pe'].iloc[-1]]])

In [5]:
#Exctracting information from nuclear recoil response
s2_energies = resp_nr['energy_kev'].values
bin_starts = resp_nr['energy_bin_start_kev'].values
bin_ends = resp_nr['energy_bin_end_kev'].values
dE = bin_ends - bin_starts

response_matrix_nr = resp_nr.values[:,3:] #we start from the 4th column, since the 3 previous ones are energies. The 4th column is s2_bin_000.

In [6]:
import wimprates as wr
reference_cross_section = 1e-45  # cm^2
rate_pertonneyearkev = wr.rate_wimp_std(
    es=s2_energies, 
    mw=4, 
    sigma_nucleon=reference_cross_section)

c:\Users\ana\DM1StatsGroupRepo2026\venv\Lib\site-packages\wimprates\__init__.py:6: UserWarning: Default WIMP parameters are changed in accordance with https://arxiv.org/abs/2105.00599 (github.com/JelleAalbers/wimprates/pull/14)
  warnings.warn(
c:\Users\ana\DM1StatsGroupRepo2026\venv\Lib\site-packages\wimprates\utils.py:40: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f)


In [7]:
#Most of the code below was copied directly from the Public Data Release
rate_pertonneyear = rate_pertonneyearkev * dE
rate_before_cutoff = rate_pertonneyear * 0.97678 
#The authors of the paper remove events below 0.7keV
recoil_energy_cutoff_kev = 0.7
rate_after_cutoff = rate_before_cutoff.copy()

# Which bin contains the cutoff?
cutoff_bin_index = (bin_starts < recoil_energy_cutoff_kev).sum() - 1 #This counts how many bins start below 0.7 keV, then subtracts 1 to get the index

# All bins fully below 0.7 keV are removed
rate_after_cutoff[:cutoff_bin_index] = 0

# Suppress the spectrum proportionally in the bin with the cutoff
suppress_by = (
    (recoil_energy_cutoff_kev - bin_starts[cutoff_bin_index]) 
    / bin_starts[cutoff_bin_index])
assert 0 <= suppress_by <= 1

#Only keep the part above the cutoff
rate_after_cutoff[cutoff_bin_index] *= 1 - suppress_by 

In [8]:
b_er = bg['er_background_events']
b_cevns = bg['cevns_background_events']
b_nominal = b_er + b_cevns
k_obs, _ = np.histogram(events['s2_area_pe'].values, bins=s2_bin_edges)

s_i = rate_after_cutoff @ response_matrix_nr
b_i = b_nominal.values

Now we add a signal to the model

We define the joint likelihood function $\mathcal{L}(\text{data} \mid \mu_s, \beta)$ to account for both the dark matter signal strength and the background normalization. However, now we use Bayesian Inference, which treats both the signal strength $\mu_s$ and background normalization $\beta$ as random variables.

We are using the same Region of Interest used for the Frequentist Likelihood (which treated $\mu_s$ as a fixed parameter).

In [9]:
#Reusing the ROI
roi = (165.3, 271.7)
roi_mask = (s2_bin_centers_log >= roi[0]) & (s2_bin_centers_log <= roi[1])

# Redefine μ_i to accept both μ_s and β as arguments
def mu_i_combined(mu_s, beta):
    """
    Expected events per bin, using the ROI mask defined in the likelihood file.
    """
    return beta * b_nominal.values[roi_mask] + mu_s * s_i[roi_mask]

#Log-likelihood
def log_likelihood_combined(mu_s, beta):
    """
    Joint log-likelihood for signal strength (μ_s) and background normalization (β).
    """
    expected = mu_i_combined(mu_s, beta)
    expected = np.clip(expected, 1e-12, None)
    return np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

# Full Likelihood (L = exp(logL)) for integration
def likelihood_combined(mu_s, beta):
    return np.exp(log_likelihood_combined(mu_s, beta))

BAYES FACTOR

Now that we have a likelihood that is a function of both $\mu_s$ and $\beta$, we can calculate the Bayes factor. For that, we need to calculate the evidence of model 0 (only background) $P(data \mid H_0)$, and of model 1 (background+signal) $P(data \mid H_1)$.

We will calculate the Bayes factor for two different priors: flat (uniform) prior and scale invariant prior. In this way, we will study the sensitivity of Bayes factor to prior.

We are starting with a flat prior, and the evidence can be calculated as the integral of the likelihood multiplied by the prior over the parameter space.

In [13]:
# Range definitions
mu_s_values = np.linspace(0, 100, 200)
beta_values = np.linspace(0.1, 30.0, 100)

# ---------------------------------------------------------
# FLAT PRIORS
# ---------------------------------------------------------
# Pi(beta) = 1, Pi(mu_s) = 1
# P(data | H0)_flat
likes_H0_flat = np.array([likelihood_combined(0, b) * 1.0 for b in beta_values])
P_data_H0_flat = trapezoid(likes_H0_flat, beta_values)

# P(data | H1)_flat
evidence_grid_H1_flat = np.zeros((len(mu_s_values), len(beta_values)))
for i, m in enumerate(mu_s_values):
    for j, b in enumerate(beta_values):
        evidence_grid_H1_flat[i, j] = likelihood_combined(m, b) * 1.0 * 1.0

P_data_H1_flat = trapezoid(trapezoid(evidence_grid_H1_flat, beta_values, axis=1), mu_s_values)
BF_01_flat = P_data_H0_flat / P_data_H1_flat
print (f"Bayes Factor (Flat Priors): {BF_01_flat:.4f}")

Bayes Factor (Flat Priors): 0.0089


Then, for the scale invariant prior:

In [ ]:
# ---------------------------------------------------------
# CASE B: SCALE INVARIANT PRIORS
# ---------------------------------------------------------
# Pi(beta) = 1/beta, Pi(mu_s) = 1/(mu_s + eps)
eps = 1e-3
def pi_si_beta(b): return 1.0 / b
def pi_si_mu(m): return 1.0 / (m + eps)

# P(data | H0)_SI
likes_H0_SI = np.array([likelihood_combined(0, b) * pi_si_beta(b) for b in beta_values])
P_data_H0_SI = trapezoid(likes_H0_SI, beta_values)

# P(data | H1)_SI
evidence_grid_H1_SI = np.zeros((len(mu_s_values), len(beta_values)))
for i, m in enumerate(mu_s_values):
    for j, b in enumerate(beta_values):
        evidence_grid_H1_SI[i, j] = likelihood_combined(m, b) * pi_si_mu(m) * pi_si_beta(b)

P_data_H1_SI = trapezoid(trapezoid(evidence_grid_H1_SI, beta_values, axis=1), mu_s_values)
BF_01_SI = P_data_H0_SI / P_data_H1_SI

Finally, we can compare both values of the Bayes factor in function of the prior applied.

In [ ]:
# Final Comparison
print(f"Bayes Factor (Flat Priors): {BF_01_flat:.4f}")
print(f"Bayes Factor (Scale Invariant): {BF_01_SI:.4f}")